In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (silhouette_score, classification_report, confusion_matrix,
                             accuracy_score, r2_score, mean_absolute_error, root_mean_squared_error)

# ==============================================================================
# 1. CARGA DE DATOS Y CODIFICACIÓN DUMMY (ONE-HOT ENCODING)
# ==============================================================================
df = pd.read_csv('df_ml (1).csv')

# Variables categóricas indicadas e identificadores a descartar
cat_cols = ['tam_loc', 'est_socio', 'clase_hog', 'sexo_jefe', 'educa_jefe']
id_cols = ['folioviv', 'foliohog', 'ubica_geo']

# Aplicar get_dummies (drop_first=True evita multicolinealidad perfecta)
df_ml = pd.get_dummies(df.drop(columns=id_cols), columns=cat_cols, drop_first=True, dtype=int)
print(f"Dimensiones tras crear dummies: {df_ml.shape}")

# ==============================================================================
# 2. MODELO NO SUPERVISADO: K-MEANS
# ==============================================================================
scaler_km = StandardScaler()
X_kmeans = scaler_km.fit_transform(df_ml)

# Método del codo para k de 2 a 7
inertias = []
K_range = range(2, 8)
for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_kmeans)
    inertias.append(km_temp.inertia_)

# Entrenamiento del modelo óptimo con k = 3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_kmeans)

print("\n=== PERFIL PROMEDIO POR CLÚSTER (K-MEANS, K=3) ===")
resumen_clusters = df.groupby('cluster_kmeans')[
    ['ing_cor', 'edad_jefe', 'tot_integ', 'ocupados', 'percep_ing', 'menores', 'p65mas']
].mean().round(2)
resumen_clusters['total_hogares'] = df['cluster_kmeans'].value_counts()
print(resumen_clusters)

# ==============================================================================
# 3. MODELO SUPERVISADO: K-NEAREST NEIGHBORS (KNN)
# ==============================================================================
# Separar matriz predictora (X) y variable objetivo (y = ing_cor)
X = df_ml.drop(columns=['ing_cor'])
y_continuo = df_ml['ing_cor']

# Discretización de ing_cor para KNN Clasificador:
# A) Binaria (Por debajo vs. Por encima de la mediana: $51,577.31)
y_binario = np.where(y_continuo >= y_continuo.median(), 'Ingreso Alto (>=Mediana)', 'Ingreso Bajo (<Mediana)')

# B) Multiclase por Terciles (Bajo, Medio, Alto)
y_terciles = pd.qcut(y_continuo, q=3, labels=['Ingreso Bajo', 'Ingreso Medio', 'Ingreso Alto'])

# División Train / Test (80% entrenamiento, 20% prueba)
X_train, X_test, y_bin_tr, y_bin_te, y_ter_tr, y_ter_te, y_reg_tr, y_reg_te = train_test_split(
    X, y_binario, y_terciles, y_continuo, test_size=0.20, random_state=42, stratify=y_terciles
)

# Estandarización ajustada únicamente con el conjunto de entrenamiento (evita Data Leakage)
scaler_knn = StandardScaler()
X_train_scaled = scaler_knn.fit_transform(X_train)
X_test_scaled = scaler_knn.transform(X_test)

# --- 3.1 KNN Clasificador Binario (2 Clases) ---
knn_bin = KNeighborsClassifier(n_neighbors=31, weights='uniform', n_jobs=-1)
knn_bin.fit(X_train_scaled, y_bin_tr)
y_bin_pred = knn_bin.predict(X_test_scaled)

print("\n=== RESULTADOS KNN CLASIFICADOR BINARIO (MEDIANA DE ING_COR) ===")
print(f"Accuracy: {accuracy_score(y_bin_te, y_bin_pred):.4f}")
print(classification_report(y_bin_te, y_bin_pred))

# --- 3.2 KNN Clasificador Multiclase (3 Terciles de Ingreso) ---
knn_ter = KNeighborsClassifier(n_neighbors=31, weights='uniform', n_jobs=-1)
knn_ter.fit(X_train_scaled, y_ter_tr)
y_ter_pred = knn_ter.predict(X_test_scaled)

print("\n=== RESULTADOS KNN CLASIFICADOR MULTICLASE (TERCILES DE ING_COR) ===")
print(f"Accuracy: {accuracy_score(y_ter_te, y_ter_pred):.4f}")
print(classification_report(y_ter_te, y_ter_pred))

# --- 3.3 KNN Regresor (Predicción directa de ing_cor continuo) ---
knn_reg = KNeighborsRegressor(n_neighbors=31, weights='uniform', n_jobs=-1)
knn_reg.fit(X_train_scaled, y_reg_tr)
y_reg_pred = knn_reg.predict(X_test_scaled)

print("\n=== RESULTADOS KNN REGRESOR (ING_COR CONTINUO) ===")
print(f"R² Score: {r2_score(y_reg_te, y_reg_pred):.4f}")
print(f"MAE:      ${mean_absolute_error(y_reg_te, y_reg_pred):,.2f} MXN")
print(f"RMSE:     ${root_mean_squared_error(y_reg_te, y_reg_pred):,.2f} MXN")

Dimensiones tras crear dummies: (83454, 30)

=== PERFIL PROMEDIO POR CLÚSTER (K-MEANS, K=3) ===
                 ing_cor  edad_jefe  tot_integ  ocupados  percep_ing  menores  \
cluster_kmeans                                                                  
0               75035.57      53.56       5.45      2.76        3.49     1.00   
1               57058.12      43.96       3.27      1.50        1.89     0.75   
2               43845.47      64.28       1.59      0.70        1.43     0.03   

                p65mas  total_hogares  
cluster_kmeans                         
0                 0.37          18741  
1                 0.07          42741  
2                 0.86          21972  

=== RESULTADOS KNN CLASIFICADOR BINARIO (MEDIANA DE ING_COR) ===
Accuracy: 0.7276
                          precision    recall  f1-score   support

Ingreso Alto (>=Mediana)       0.74      0.72      0.73      8406
 Ingreso Bajo (<Mediana)       0.72      0.74      0.73      8285

               